<a href="https://colab.research.google.com/github/subudear/deep-learning/blob/main/assignment1/DinoV2/assignment1_dinov2_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DINOv2-ViT-B/14 Cell Culture Classifier
Full pipeline with focal loss, extreme-minority oversampling, and 3-phase progressive fine-tuning.

In [ ]:
# STEP 1: Install packages & imports + GPU check
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# STEP 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# STEP 3: Extract dataset zip files from Drive

In [ ]:
dataset_path = '/content/drive/MyDrive/compx525assign1/cell_cultures'
extract_path = '/content/dataset'

os.makedirs(extract_path, exist_ok=True)

for f in os.listdir(dataset_path):
    if f.endswith('.zip'):
        zip_path = os.path.join(dataset_path, f)
        print('Extracting:', zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)

print('Extraction complete')
print('Extracted folders:', os.listdir(extract_path))

# STEP 4: Define paths & audit dataset
Categorises classes into tiers: empty, extreme minority, moderate minority, majority.

In [ ]:
train_dir  = '/content/dataset/train'
test_dir   = '/content/dataset/test'
image_exts = ('.jpg', '.jpeg', '.png')

print('=== DATASET AUDIT ===')
class_counts = {}
for cls in sorted(os.listdir(train_dir)):
    cls_path = os.path.join(train_dir, cls)
    if not os.path.isdir(cls_path):
        continue
    n = sum(f.lower().endswith(image_exts) for f in os.listdir(cls_path))
    class_counts[cls] = n
    tier = 'EMPTY' if n == 0 else 'EXTREME (<15)' if n < 15 else 'MINORITY (<100)' if n < 100 else 'OK'
    print(f'  {cls}: {n:5d}  [{tier}]')

test_imgs = [f for f in os.listdir(test_dir) if f.lower().endswith(image_exts)]
print(f'\nTotal classes: {len(class_counts)}')
print(f'Total train images: {sum(class_counts.values())}')
print(f'Total test images:  {len(test_imgs)}')

# Tier categorisation — these drive ALL downstream decisions
EMPTY_CLASSES    = [c for c, n in class_counts.items() if n == 0]
EXTREME_CLASSES  = [c for c, n in class_counts.items() if 0 < n < 15]
MINORITY_CLASSES = [c for c, n in class_counts.items() if 15 <= n < 100]

print(f'\nEmpty (excluded from training): {EMPTY_CLASSES}')
print(f'Extreme minority (train only, heavy oversample): {EXTREME_CLASSES}')
print(f'Moderate minority (heavier augmentation): {MINORITY_CLASSES}')

# STEP 5: Build training dataframe (exclude empty classes)

In [ ]:
rows = []
for cls, n in class_counts.items():
    if cls in EMPTY_CLASSES:
        print(f'  Skipping class [{cls}] — 0 images')
        continue
    cls_path = os.path.join(train_dir, cls)
    for fname in os.listdir(cls_path):
        if fname.lower().endswith(image_exts):
            rows.append({'filepath': os.path.join(cls_path, fname), 'label': cls})

train_df = pd.DataFrame(rows)
print(f'Total usable training images: {len(train_df)}')
print('\nClass counts:')
print(train_df['label'].value_counts().sort_index())

# STEP 6: Train/val split
Extreme minority classes (< 15 images) go to training ONLY — too few images for a reliable validation split.

In [ ]:
extreme_df    = train_df[train_df['label'].isin(EXTREME_CLASSES)].copy()
splittable_df = train_df[~train_df['label'].isin(EXTREME_CLASSES)].copy()

train_split_df, val_split_df = train_test_split(
    splittable_df,
    test_size=0.2,
    stratify=splittable_df['label'],
    random_state=42
)

# Extreme classes added back to training only (never validation)
train_split_df = pd.concat([train_split_df, extreme_df], ignore_index=True)

print(f'Train size (before oversample): {len(train_split_df)}')
print(f'Val size:                       {len(val_split_df)}')
print('\nTrain class counts:')
print(train_split_df['label'].value_counts().sort_index())
print('\nVal class counts:')
print(val_split_df['label'].value_counts().sort_index())

# STEP 7: Encode labels
Class j (0 images) is already excluded — label indices are compact.

In [ ]:
all_labels  = sorted(train_df['label'].unique())
label_to_idx = {label: idx for idx, label in enumerate(all_labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
num_classes  = len(label_to_idx)

print(f'Usable classes ({num_classes}): {all_labels}')
print('label_to_idx:', label_to_idx)

train_split_df = train_split_df.copy()
val_split_df   = val_split_df.copy()
train_split_df['label_idx'] = train_split_df['label'].map(label_to_idx)
val_split_df['label_idx']   = val_split_df['label'].map(label_to_idx)

# STEP 8: Oversample extreme minority classes
Row-level oversampling ensures extreme minority classes appear ~120 times per epoch.
Each appearance gets a different random augmentation, creating genuine diversity.

In [ ]:
EXTREME_TARGET = 120  # target appearances per epoch for extreme minority classes

parts = []
for cls, grp in train_split_df.groupby('label'):
    n = len(grp)
    if cls in EXTREME_CLASSES:
        repeats = max(1, EXTREME_TARGET // n)
        repeated = pd.concat([grp] * repeats, ignore_index=True)
        parts.append(repeated)
        print(f'  [{cls}] {n} images -> {len(repeated)} rows (x{repeats} oversample)')
    else:
        parts.append(grp)

train_split_df = (
    pd.concat(parts, ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print(f'\nTrain size (after oversample): {len(train_split_df)}')
print('Class counts after oversample:')
print(train_split_df['label'].value_counts().sort_index())

# STEP 9: Image transforms
DINOv2-ViT-B/14 uses 14px patches. 448px = 32x14 patches — preserves cell morphology detail.
All flips and rotations are valid: microscopy cell images have no canonical orientation.

In [ ]:
DINOV2_MEAN = [0.485, 0.456, 0.406]
DINOV2_STD  = [0.229, 0.224, 0.225]
IMG_SIZE    = 448  # 32 x 14px patches — reduce to 336 if OOM

# Standard training augmentation (all classes)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.15),
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=DINOV2_MEAN, std=DINOV2_STD),
])

# Heavier augmentation for extreme minority classes (maximise diversity from few images)
extreme_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2),
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.3),
    transforms.RandomApply([transforms.RandomPerspective(distortion_scale=0.2)], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=DINOV2_MEAN, std=DINOV2_STD),
])

# Validation: deterministic center crop (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=DINOV2_MEAN, std=DINOV2_STD),
])

print(f'Input size: {IMG_SIZE}x{IMG_SIZE}')
print(f'Patches per image: {(IMG_SIZE // 14) ** 2} (DINOv2 14px patch size)')

# STEP 10: Dataset, DataLoader & Focal Loss
Focal Loss (gamma=2) focuses training on hard/misclassified examples — much better than weighted
cross-entropy for extreme 320:1 class imbalance. Sqrt weighting softens oversampled counts.

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, df, transform=None, extreme_transform=None, extreme_classes=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.extreme_transform = extreme_transform
        self.extreme_classes = set(extreme_classes or [])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['filepath']).convert('RGB')
        label = int(row['label_idx'])
        if row['label'] in self.extreme_classes and self.extreme_transform is not None:
            image = self.extreme_transform(image)
        elif self.transform is not None:
            image = self.transform(image)
        return image, label


class FocalLoss(nn.Module):
    """
    Focal loss: down-weights easy examples, focuses on hard/misclassified ones.
    gamma=2 is the standard setting. class weight handles baseline frequency skew.
    """
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        if weight is not None:
            self.register_buffer('weight', weight)
        else:
            self.weight = None

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        ce_smooth = (1 - self.label_smoothing) * ce + self.label_smoothing * (-torch.log(pt + 1e-8)).mean()
        return ((1 - pt) ** self.gamma * ce_smooth).mean()


# Build datasets
train_dataset = ImageDataset(
    train_split_df,
    transform=train_transform,
    extreme_transform=extreme_train_transform,
    extreme_classes=EXTREME_CLASSES,
)
val_dataset = ImageDataset(val_split_df, transform=val_transform)

# Sqrt weighting softens the effect of oversampled extreme-minority rows
counts = train_split_df['label_idx'].value_counts().sort_index()
focal_weights = 1.0 / torch.tensor(counts.values, dtype=torch.float).sqrt()
focal_weights = (focal_weights / focal_weights.sum() * num_classes).to(device)
print('Focal loss class weights:', focal_weights)

BATCH_SIZE = 8   # DINOv2-B at 518x518 on T4 � halved from 16 to fit VRAM
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

# STEP 11: Load DINOv2-ViT-B/14 + classifier head
DINOv2 produces a 768-dim CLS token from 14px patch attention — far superior to ResNet
convolutions for fine-grained cell morphology classification.

In [ ]:
print('Loading DINOv2-ViT-B/14 from torch.hub ...')
backbone = torch.hub.load(
    'facebookresearch/dinov2',
    'dinov2_vitb14',
    pretrained=True,
)


class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.5):  # raised from 0.3 → 0.5
        super().__init__()
        self.backbone = backbone
        embed_dim = backbone.embed_dim  # 768 for ViT-B
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, x):
        cls_token = self.backbone(x)  # [B, 768] — CLS token
        return self.head(cls_token)


model = DINOv2Classifier(backbone, num_classes=num_classes).to(device)

print(f'Backbone embed_dim: {backbone.embed_dim}')
print(f'Num transformer blocks: {len(backbone.blocks)}')
print(f'Classifier head: {model.head}')
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')

best_model_path = '/content/drive/MyDrive/dinov2_best_model_v2.pth'


# STEP 12: Phase 1 — Train classifier head only (5 epochs)
Backbone frozen. Initialises head weights before any backbone updates.
Mixed-precision (AMP) used throughout to fit DINOv2 in Colab VRAM.

In [ ]:
# Freeze backbone, train head only
for param in model.backbone.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True

criterion  = FocalLoss(gamma=2.0, weight=focal_weights, label_smoothing=0.1)
optimizer  = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
scaler     = torch.amp.GradScaler('cuda')

num_epochs   = 5
best_val_acc = 0.0

print(f'Phase 1 trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    train_loss = running_loss / len(train_loader)
    train_acc  = correct / total

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)
    val_acc = val_correct / val_total

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state_dict': model.state_dict(), 'label_to_idx': label_to_idx,
                    'idx_to_label': idx_to_label, 'best_val_acc': best_val_acc}, best_model_path)
        print(f'  --> Saved best model (val_acc={val_acc:.4f})')

    print(f'Phase 1 | Epoch {epoch+1}/{num_epochs} | Loss: {train_loss:.4f} | '
          f'Train: {train_acc:.4f} | Val: {val_acc:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}')

print(f'Phase 1 best val acc: {best_val_acc:.4f}')


# STEP 13: Phase 2 — Unfreeze last 4 transformer blocks + head (20 epochs)
DINOv2-ViT-B has 12 blocks (0–11). Blocks 8–11 are unfrozen to adapt deep features
to cell morphology. ReduceLROnPlateau — LR only drops when val acc stops improving.

In [ ]:
# Freeze everything first, then selectively unfreeze
for param in model.backbone.parameters():
    param.requires_grad = False

# Unfreeze last 4 transformer blocks (blocks 8, 9, 10, 11)
for block in model.backbone.blocks[-4:]:
    for param in block.parameters():
        param.requires_grad = True

# Unfreeze final norm layer
for param in model.backbone.norm.parameters():
    param.requires_grad = True

# Head always trainable
for param in model.head.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2 trainable params: {trainable:,}')

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-5,
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3,
)
scaler = torch.amp.GradScaler('cuda')

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc  = correct / total

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)
    val_acc = val_correct / val_total

    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]['lr']

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state_dict': model.state_dict(), 'label_to_idx': label_to_idx,
                    'idx_to_label': idx_to_label, 'best_val_acc': best_val_acc}, best_model_path)
        print(f'  --> Saved best model at epoch {epoch+1} (val_acc={val_acc:.4f})')

    print(f'Phase 2 | Epoch {epoch+1}/{num_epochs} | Loss: {train_loss:.4f} | '
          f'Train: {train_acc:.4f} | Val: {val_acc:.4f} | LR: {current_lr:.2e}')

print(f'Phase 2 best val acc: {best_val_acc:.4f}')


# STEP 14: Phase 3 — Full model fine-tune (15 epochs)
Loads best Phase 2 checkpoint. All layers unfrozen at lr=5e-6.
ReduceLROnPlateau with patience=4 gives more room before cutting LR.

In [ ]:
# Load best model from phases 1+2 as starting point
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
print(f'Starting Phase 3 from best val acc: {checkpoint["best_val_acc"]:.4f}')

# Unfreeze everything
for param in model.parameters():
    param.requires_grad = True
print(f'Phase 3 trainable params: {sum(p.numel() for p in model.parameters()):,}')

# Raised weight_decay 1e-4 → 5e-4 to combat the 23% train-val gap
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6, weight_decay=5e-4)
# Reduced patience 4 → 2: LR needs to respond faster over only 15 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2,
)
scaler = torch.amp.GradScaler('cuda')

num_epochs = 15

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc  = correct / total

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)
    val_acc = val_correct / val_total

    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]['lr']

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state_dict': model.state_dict(), 'label_to_idx': label_to_idx,
                    'idx_to_label': idx_to_label, 'best_val_acc': best_val_acc}, best_model_path)
        print(f'  --> Saved best model at phase3 epoch {epoch+1} (val_acc={val_acc:.4f})')

    print(f'Phase 3 | Epoch {epoch+1}/{num_epochs} | Loss: {train_loss:.4f} | '
          f'Train: {train_acc:.4f} | Val: {val_acc:.4f} | LR: {current_lr:.2e}')

print(f'Best val acc after Phase 3: {best_val_acc:.4f}')
print(f'Best model saved to: {best_model_path}')


# STEP 15: Validation evaluation — confusion matrix & classification report

In [ ]:
# Reload best checkpoint (in case best was at an earlier epoch than final)
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Evaluating best model (val_acc={checkpoint["best_val_acc"]:.4f})')

all_true, all_pred = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            outputs = model(images)
        all_true.extend(labels.numpy())
        all_pred.extend(outputs.argmax(1).cpu().numpy())

class_names = [idx_to_label[i] for i in range(num_classes)]
cm = confusion_matrix(all_true, all_pred)

plt.figure(figsize=(12, 10))
plt.imshow(cm, interpolation='nearest')
plt.title('Validation Confusion Matrix (DINOv2-ViT-B/14)')
plt.colorbar()
tick_marks = np.arange(num_classes)
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

print(classification_report(all_true, all_pred, target_names=class_names, zero_division=0))

# STEP 16: Build test dataframe & DataLoader

In [ ]:
test_rows = []
for fname in sorted(os.listdir(test_dir)):
    if fname.lower().endswith(image_exts):
        test_rows.append({'filepath': os.path.join(test_dir, fname), 'filename': fname})

test_df = pd.DataFrame(test_rows)
print(f'Test images: {len(test_df)}')
print(test_df.head())


class TestImageDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['filepath']).convert('RGB')
        return self.transform(image), row['filename']


test_dataset = TestImageDataset(test_df, val_transform)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'Test loader ready — {len(test_loader)} batches')

# STEP 17: Generate final submission CSV

In [ ]:
model.eval()
all_filenames, all_pred_labels = [], []

with torch.no_grad():
    for images, filenames in test_loader:
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            outputs = model(images)
        preds = outputs.argmax(1).cpu()
        all_filenames.extend(filenames)
        all_pred_labels.extend([idx_to_label[p.item()] for p in preds])

submission_df = pd.DataFrame({'TestFileName': all_filenames, 'Class': all_pred_labels})

csv_path = '/content/drive/MyDrive/dinov2_submission_final.csv'
submission_df.to_csv(csv_path, index=False)

print(submission_df.head(10))
print(f'\nTotal predictions: {len(submission_df)}')
print(f'Prediction distribution:\n{submission_df["Class"].value_counts().sort_index()}')
print(f'\nSaved to: {csv_path}')